12/6/2025 - Moosa

Purpose: The goal is to understand how each model performs on unseen data and save all evaluation outputs to the correct project folders for team reference and presentation.

In [1]:
import sys; 
import pandas as pd
import pickle
import os
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
sys.path.append("../../") #To allow imports from parent folders

# Load
df = pd.read_csv("../data/cleaned/wednesday_cleaned.csv")
X = df.drop(['Label', 'Attack'], axis=1)
y = df['Attack']

# Train/test split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale test set
scaler = StandardScaler()
scaler.fit(X_train)
X_test_scaled = scaler.transform(X_test)

# Models loading
models = {
    "random_forest": "../models/saved_model/random_forest.pkl",
    "decision_tree": "../models/saved_model/decision_tree.pkl",
    "svm": "../models/saved_model/svm.pkl"
}

loaded = {}
for name, path in models.items():
    with open(path, "rb") as f:
        loaded[name] = pickle.load(f)

# Output folders
os.makedirs("../models/evaluation/confusion_matrices", exist_ok=True)
os.makedirs("../models/evaluation/metrics_tables", exist_ok=True)

# Evaluate and save outputs
for name, model in loaded.items():
    print(f"\n{name.upper()} \n")

    y_pred = model.predict(X_test_scaled)

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    print("\nConfusion Matrix:\n", cm)

    with open(f"../models/evaluation/confusion_matrices/{name}_cm.txt", "w") as f:
        f.write(str(cm))

    # Classification report
    report = classification_report(y_test, y_pred)
    print("\nClassification Report:\n", report)

    with open(f"../models/evaluation/metrics_tables/{name}_report.txt", "w") as f:
        f.write(report)

print("\nAll evaluations completed and saved!!")


/Users/usid/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/usid/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/usid/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.7.1 


RANDOM_FOREST 


Confusion Matrix:
 [[  998     1]
 [    1 11201]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       999
           1       1.00      1.00      1.00     11202

    accuracy                           1.00     12201
   macro avg       1.00      1.00      1.00     12201
weighted avg       1.00      1.00      1.00     12201


DECISION_TREE 


Confusion Matrix:
 [[  996     3]
 [    2 11200]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       999
           1       1.00      1.00      1.00     11202

    accuracy                           1.00     12201
   macro avg       1.00      1.00      1.00     12201
weighted avg       1.00      1.00      1.00     12201


SVM 


Confusion Matrix:
 [[  943    56]
 [    5 11197]]

Classification Report:
               precision    recall  f1-score   support

           0       0.9

Evaluated three complex models (Random Forest, Decision Tree, and SVM) using the test set and generated both confusion matrices and classification reports. All models performed extremely well overall, with Random Forest achieving perfect accuracy and the other two models showing only a few misclassifications. 

The confusion matrices were saved in models/evaluation/confusion_matrices/ and the classification reports were saved in models/evaluation/metrics_tables/ for the team to use in later steps.

## Task Complete

---

12/6/2025 - Umar

generating performance summary table for all models

In [2]:
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

all_models = loaded.copy()  

# load simple models
nb_path = '../models/saved_model/naive_bayes.pkl'
knn_path = '../models/saved_model/knn.pkl'
lr_path = '../models/saved_model/logistic_regression.pkl'

with open(nb_path, 'rb') as f:
    all_models['naive_bayes'] = pickle.load(f)
print('loaded naive bayes')

with open(knn_path, 'rb') as f:
    all_models['knn'] = pickle.load(f)
print('loaded knn')

with open(lr_path, 'rb') as f:
    all_models['logistic_regression'] = pickle.load(f)
print('loaded logistic regression')

print(f'total: {len(all_models)} models')

loaded naive bayes
loaded knn
loaded logistic regression
total: 6 models


In [ ]:
# calculate metrics accuracy, recall, f1, auc
performance_data = []

for model_name, model in all_models.items():
    preds = model.predict(X_test_scaled)
    
    # need probabilities for auc
    # some models have predict_proba, svm uses decision_function
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test_scaled)[:, 1]
        auc = roc_auc_score(y_test, proba)
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X_test_scaled)
        auc = roc_auc_score(y_test, scores)
    else:
        auc = None  
    
    # calculate other metrics
    acc = accuracy_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    
    performance_data.append({
        'model': model_name,
        'accuracy': acc,
        'recall': rec,
        'f1_score': f1,
        'roc_auc': auc
    })
    
    print(f'{model_name}: acc={acc:.4f}, rec={rec:.4f}, f1={f1:.4f}, auc={auc:.4f}')

# make dataframe
df_results = pd.DataFrame(performance_data)

# sort by f1 score
df_results = df_results.sort_values('f1_score', ascending=False)

print('\nPerformance Summary:')
print(df_results)

random_forest: acc=0.9998, rec=0.9999, f1=0.9999, auc=1.0000
decision_tree: acc=0.9996, rec=0.9998, f1=0.9998, auc=0.9984


/Users/usid/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/usid/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/usid/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/Users/usid/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/Users/usid/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feat

svm: acc=0.9950, rec=0.9996, f1=0.9973, auc=0.9979
naive_bayes: acc=0.9826, rec=0.9878, f1=0.9905, auc=0.9718
knn: acc=0.9989, rec=0.9995, f1=0.9994, auc=0.9979
logistic_regression: acc=0.9937, rec=0.9998, f1=0.9966, auc=0.9966

Performance Summary:
                 model  accuracy    recall  f1_score   roc_auc
0        random_forest  0.999836  0.999911  0.999911  0.999991
1        decision_tree  0.999590  0.999821  0.999777  0.998409
4                  knn  0.998935  0.999464  0.999420  0.997900
2                  svm  0.995000  0.999554  0.997283  0.997864
5  logistic_regression  0.993689  0.999821  0.996574  0.996625
3          naive_bayes  0.982624  0.987770  0.990511  0.971785


In [4]:
# save to csv
save_path = '../models/evaluation/metrics_tables/performance_summary.csv'
df_results.to_csv(save_path, index=False)
print(f'saved to {save_path}')

saved to ../models/evaluation/metrics_tables/performance_summary.csv


Performance summary table is done. All 6 models evaluated. 

All models have high AUC (>0.98) so they can distinguish dos attacks from benign traffic well.

Table saved as csv in metrics_tables folder.